In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 6.4 MB/s eta 0:00:00


In [2]:
from pathlib import Path

# ============================================
# BDD100K ROOT
# ============================================

BDD_ROOT = Path(
    "/kaggle/input/datasets/solesensei/solesensei_bdd100k"
)

# ============================================
# IMAGE PATHS
# ============================================

IMAGE_ROOT = BDD_ROOT / "bdd100k" / "bdd100k"/ "images"/"100k"

IMAGE_TRAIN = IMAGE_ROOT / "train"
IMAGE_VAL   = IMAGE_ROOT / "val"

# ============================================
# LABEL PATHS
# ============================================

LABEL_ROOT = (
    BDD_ROOT
    / "bdd100k_labels_release"
    / "bdd100k"
    / "labels"
)

LABEL_TRAIN = LABEL_ROOT / "bdd100k_labels_images_train.json"
LABEL_VAL   = LABEL_ROOT / "bdd100k_labels_images_val.json"

# ============================================
# OUTPUT
# ============================================

OUTPUT_ROOT = Path("/kaggle/working/bdd100k_yolo")

print("TRAIN IMAGES :", IMAGE_TRAIN)
print("VAL IMAGES   :", IMAGE_VAL)
print("TRAIN LABEL  :", LABEL_TRAIN)
print("VAL LABEL    :", LABEL_VAL)
print("OUTPUT       :", OUTPUT_ROOT)

TRAIN IMAGES : /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train
VAL IMAGES   : /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/val
TRAIN LABEL  : /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
VAL LABEL    : /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json
OUTPUT       : /kaggle/working/bdd100k_yolo


In [3]:
TRAIN_DIRS = [
    Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA"),
    Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainB"),
    Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/testA"),
    Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/testB"),
    Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train"),
]

VAL_DIRS = [
    p for p in IMAGE_ROOT.iterdir()
    if p.is_dir() and p.name.lower().startswith("val")
]




In [4]:
print("\nExistence check:")
print("Train images:", IMAGE_TRAIN.exists())
print("Val images  :", IMAGE_VAL.exists())
print("Train label :", LABEL_TRAIN.exists())
print("Val label   :", LABEL_VAL.exists())


Existence check:
Train images: True
Val images  : True
Train label : True
Val label   : True


In [5]:
import json
import shutil
from pathlib import Path
from collections import Counter

from PIL import Image
from tqdm.notebook import tqdm

In [6]:
CLASSES = [
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motor",
    "bike",
    "traffic light",
    "traffic sign",
]

CLASS_TO_ID = {
    name: idx
    for idx, name in enumerate(CLASSES)
}

CLASS_TO_ID

{'person': 0,
 'rider': 1,
 'car': 2,
 'truck': 3,
 'bus': 4,
 'train': 5,
 'motor': 6,
 'bike': 7,
 'traffic light': 8,
 'traffic sign': 9}

In [7]:
from pathlib import Path

OUTPUT_ROOT = Path("/kaggle/working/bdd100k_yolo")

TRAIN_IMAGE_OUT = OUTPUT_ROOT / "images" / "train"
VAL_IMAGE_OUT   = OUTPUT_ROOT / "images" / "val"

TRAIN_LABEL_OUT = OUTPUT_ROOT / "labels" / "train"
VAL_LABEL_OUT   = OUTPUT_ROOT / "labels" / "val"

for path in [
    TRAIN_IMAGE_OUT,
    VAL_IMAGE_OUT,
    TRAIN_LABEL_OUT,
    VAL_LABEL_OUT,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_ROOT)

Output directory: /kaggle/working/bdd100k_yolo


In [8]:
import json

with open(LABEL_TRAIN, "r", encoding="utf-8") as f:
    train_annotations = json.load(f)

with open(LABEL_VAL, "r", encoding="utf-8") as f:
    val_annotations = json.load(f)

print("Train annotations:", len(train_annotations))
print("Val annotations  :", len(val_annotations))

Train annotations: 69863
Val annotations  : 10000


In [9]:
def build_image_index(directories):
    image_index = {}

    for directory in directories:
        print(f"Scanning: {directory}")

        for image_path in directory.glob("*.jpg"):
            image_index[image_path.name] = image_path

    return image_index


train_image_index = build_image_index(TRAIN_DIRS)
val_image_index = build_image_index(VAL_DIRS)

print("Train:", len(train_image_index))
print("Val  :", len(val_image_index))

Scanning: /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA
Scanning: /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainB
Scanning: /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/testA
Scanning: /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/testB
Scanning: /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train
Scanning: /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/val
Train: 70000
Val  : 10000


In [10]:
def bdd_to_yolo(x1, y1, x2, y2, image_width, image_height):
    """
    BDD100K:
        x1, y1, x2, y2

    YOLO:
        x_center, y_center, width, height
    """

    x1 = max(0.0, min(float(x1), image_width))
    x2 = max(0.0, min(float(x2), image_width))

    y1 = max(0.0, min(float(y1), image_height))
    y2 = max(0.0, min(float(y2), image_height))

    if x2 <= x1 or y2 <= y1:
        return None

    x_center = ((x1 + x2) / 2.0) / image_width
    y_center = ((y1 + y2) / 2.0) / image_height

    width = (x2 - x1) / image_width
    height = (y2 - y1) / image_height

    return (
        x_center,
        y_center,
        width,
        height
    )

In [11]:
import shutil
from PIL import Image


def convert_split(
    annotations,
    image_index,
    image_output_dir,
    label_output_dir,
    split_name
):
    image_output_dir.mkdir(parents=True, exist_ok=True)
    label_output_dir.mkdir(parents=True, exist_ok=True)

    converted_images = 0
    missing_images = 0
    skipped_labels = 0
    converted_boxes = 0

    for item in annotations:

        image_name = item.get("name")

        if not image_name:
            continue

        # Tìm image từ các folder trainA/trainB hoặc valA/valB
        if image_name not in image_index:
            missing_images += 1
            continue

        source_image = image_index[image_name]

        destination_image = image_output_dir / image_name
        destination_label = label_output_dir / (
            Path(image_name).stem + ".txt"
        )

        # Đọc kích thước ảnh
        try:
            with Image.open(source_image) as img:
                image_width, image_height = img.size

        except Exception as e:
            print(f"Cannot read image: {source_image}")
            print(e)
            continue

        # Copy image
        if not destination_image.exists():
            shutil.copy2(
                source_image,
                destination_image
            )

        yolo_lines = []

        for label in item.get("labels", []):

            category = label.get("category")

            # Bỏ category không nằm trong 10 class
            if category not in CLASS_TO_ID:
                skipped_labels += 1
                continue

            box2d = label.get("box2d")

            if not box2d:
                skipped_labels += 1
                continue

            x1 = box2d.get("x1")
            y1 = box2d.get("y1")
            x2 = box2d.get("x2")
            y2 = box2d.get("y2")

            if None in [x1, y1, x2, y2]:
                skipped_labels += 1
                continue

            bbox = bdd_to_yolo(
                x1,
                y1,
                x2,
                y2,
                image_width,
                image_height
            )

            if bbox is None:
                skipped_labels += 1
                continue

            xc, yc, w, h = bbox

            class_id = CLASS_TO_ID[category]

            yolo_lines.append(
                f"{class_id} "
                f"{xc:.6f} "
                f"{yc:.6f} "
                f"{w:.6f} "
                f"{h:.6f}"
            )

            converted_boxes += 1

        # Ghi label
        with open(
            destination_label,
            "w",
            encoding="utf-8"
        ) as f:
            f.write("\n".join(yolo_lines))

        converted_images += 1

        if converted_images % 1000 == 0:
            print(
                f"[{split_name}] "
                f"{converted_images:,} images converted..."
            )

    print()
    print(f"===== {split_name.upper()} DONE =====")
    print(f"Images converted : {converted_images:,}")
    print(f"Missing images   : {missing_images:,}")
    print(f"Boxes converted  : {converted_boxes:,}")
    print(f"Labels skipped   : {skipped_labels:,}")

    return {
        "images": converted_images,
        "missing_images": missing_images,
        "boxes": converted_boxes,
        "skipped_labels": skipped_labels,
    }

In [12]:
train_stats = convert_split(
    annotations=train_annotations,
    image_index=train_image_index,
    image_output_dir=TRAIN_IMAGE_OUT,
    label_output_dir=TRAIN_LABEL_OUT,
    split_name="train"
)

[train] 1,000 images converted...
[train] 2,000 images converted...
[train] 3,000 images converted...
[train] 4,000 images converted...
[train] 5,000 images converted...
[train] 6,000 images converted...
[train] 7,000 images converted...
[train] 8,000 images converted...
[train] 9,000 images converted...
[train] 10,000 images converted...
[train] 11,000 images converted...
[train] 12,000 images converted...
[train] 13,000 images converted...
[train] 14,000 images converted...
[train] 15,000 images converted...
[train] 16,000 images converted...
[train] 17,000 images converted...
[train] 18,000 images converted...
[train] 19,000 images converted...
[train] 20,000 images converted...
[train] 21,000 images converted...
[train] 22,000 images converted...
[train] 23,000 images converted...
[train] 24,000 images converted...
[train] 25,000 images converted...
[train] 26,000 images converted...
[train] 27,000 images converted...
[train] 28,000 images converted...
[train] 29,000 images convert

In [13]:
val_stats = convert_split(
    annotations=val_annotations,
    image_index=val_image_index,
    image_output_dir=VAL_IMAGE_OUT,
    label_output_dir=VAL_LABEL_OUT,
    split_name="val"
)

[val] 1,000 images converted...
[val] 2,000 images converted...
[val] 3,000 images converted...
[val] 4,000 images converted...
[val] 5,000 images converted...
[val] 6,000 images converted...
[val] 7,000 images converted...
[val] 8,000 images converted...
[val] 9,000 images converted...
[val] 10,000 images converted...

===== VAL DONE =====
Images converted : 10,000
Missing images   : 0
Boxes converted  : 185,526
Labels skipped   : 93,711


In [14]:
def check_image_label_pairs(image_dir, label_dir, split_name):
    image_files = {
        p.stem
        for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    }

    label_files = {
        p.stem
        for p in label_dir.iterdir()
        if p.suffix.lower() == ".txt"
    }

    missing_labels = image_files - label_files
    orphan_labels = label_files - image_files

    print(f"===== {split_name.upper()} CHECK =====")
    print(f"Images : {len(image_files):,}")
    print(f"Labels : {len(label_files):,}")
    print(f"Missing labels : {len(missing_labels):,}")
    print(f"Orphan labels  : {len(orphan_labels):,}")

    if missing_labels:
        print("\nExample missing labels:")
        for x in list(missing_labels)[:10]:
            print(x)

    if orphan_labels:
        print("\nExample orphan labels:")
        for x in list(orphan_labels)[:10]:
            print(x)

    return missing_labels, orphan_labels


train_missing, train_orphan = check_image_label_pairs(
    TRAIN_IMAGE_OUT,
    TRAIN_LABEL_OUT,
    "train"
)

val_missing, val_orphan = check_image_label_pairs(
    VAL_IMAGE_OUT,
    VAL_LABEL_OUT,
    "val"
)

===== TRAIN CHECK =====
Images : 69,863
Labels : 69,863
Missing labels : 0
Orphan labels  : 0
===== VAL CHECK =====
Images : 10,000
Labels : 10,000
Missing labels : 0
Orphan labels  : 0


In [15]:
NUM_CLASSES = 10
def validate_yolo_labels(label_dir):
    invalid = []
    total_boxes = 0

    for label_file in label_dir.glob("*.txt"):

        with open(label_file, "r", encoding="utf-8") as f:
            lines = f.readlines()

        for line_no, line in enumerate(lines, start=1):

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:
                invalid.append(
                    (label_file.name, line_no, "wrong number of values")
                )
                continue

            try:
                class_id = int(parts[0])

                xc = float(parts[1])
                yc = float(parts[2])
                w = float(parts[3])
                h = float(parts[4])

            except ValueError:
                invalid.append(
                    (label_file.name, line_no, "non-numeric value")
                )
                continue

            if not (0 <= class_id < NUM_CLASSES):
                invalid.append(
                    (label_file.name, line_no, "invalid class id")
                )

            if not (0 <= xc <= 1):
                invalid.append(
                    (label_file.name, line_no, "xc out of range")
                )

            if not (0 <= yc <= 1):
                invalid.append(
                    (label_file.name, line_no, "yc out of range")
                )

            if not (0 < w <= 1):
                invalid.append(
                    (label_file.name, line_no, "width out of range")
                )

            if not (0 < h <= 1):
                invalid.append(
                    (label_file.name, line_no, "height out of range")
                )

            total_boxes += 1

    print(f"Directory: {label_dir}")
    print(f"Total boxes: {total_boxes:,}")
    print(f"Invalid entries: {len(invalid):,}")

    if invalid:
        print("\nExamples:")
        for item in invalid[:20]:
            print(item)

    return invalid


train_invalid = validate_yolo_labels(TRAIN_LABEL_OUT)
val_invalid = validate_yolo_labels(VAL_LABEL_OUT)

Directory: /kaggle/working/bdd100k_yolo/labels/train
Total boxes: 1,286,871
Invalid entries: 0
Directory: /kaggle/working/bdd100k_yolo/labels/val
Total boxes: 185,526
Invalid entries: 0


In [16]:
import yaml

data_yaml = {
    "path": str(OUTPUT_ROOT),

    "train": "images/train",
    "val": "images/val",

    "nc": NUM_CLASSES,

    "names": CLASSES,
}

DATA_YAML = OUTPUT_ROOT / "data.yaml"

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print(DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as f:
    print(f.read())

/kaggle/working/bdd100k_yolo/data.yaml
path: /kaggle/working/bdd100k_yolo
train: images/train
val: images/val
nc: 10
names:
- person
- rider
- car
- truck
- bus
- train
- motor
- bike
- traffic light
- traffic sign



In [17]:
from collections import Counter

def count_classes(label_dir):
    counter = Counter()

    for label_file in label_dir.glob("*.txt"):

        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:

                line = line.strip()

                if not line:
                    continue

                class_id = int(line.split()[0])

                if 0 <= class_id < NUM_CLASSES:
                    counter[class_id] += 1

    return counter


train_counts = count_classes(TRAIN_LABEL_OUT)
val_counts = count_classes(VAL_LABEL_OUT)

print("===== TRAIN =====")

for class_id, class_name in enumerate(CLASSES):
    print(
        f"{class_id:2d} | "
        f"{class_name:15s} | "
        f"{train_counts[class_id]:,}"
    )

print()
print("===== VAL =====")

for class_id, class_name in enumerate(CLASSES):
    print(
        f"{class_id:2d} | "
        f"{class_name:15s} | "
        f"{val_counts[class_id]:,}"
    )

===== TRAIN =====
 0 | person          | 91,349
 1 | rider           | 4,517
 2 | car             | 713,211
 3 | truck           | 29,971
 4 | bus             | 11,672
 5 | train           | 136
 6 | motor           | 3,002
 7 | bike            | 7,210
 8 | traffic light   | 186,117
 9 | traffic sign    | 239,686

===== VAL =====
 0 | person          | 13,262
 1 | rider           | 649
 2 | car             | 102,506
 3 | truck           | 4,245
 4 | bus             | 1,597
 5 | train           | 15
 6 | motor           | 452
 7 | bike            | 1,007
 8 | traffic light   | 26,885
 9 | traffic sign    | 34,908


In [18]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

model.train(
    data=DATA_YAML,
    imgsz=800,
    rect=True,
    batch=64,
    epochs=50,
    device=0,
    workers=4,

    project="/kaggle/working/yolo_bdd100k",
    name="yolo11m_800_rect",
    val = True,
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/bdd100k_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fract

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      7.02G      1.573      1.063      1.045        541        800: 100% ━━━━━━━━━━━━ 1092/1092 1.9it/s 9:40
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 1.3it/s 58.7s
                   all      10000     185526      0.414      0.276      0.272      0.139

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      7.32G      1.597       1.06      1.059        868        800: 100% ━━━━━━━━━━━━ 1092/1092 1.9it/s 9:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 1.4it/s 57.4s
                   all      10000     185526      0.519       0.29      0.283      0.143

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      7.12G      1.582      1.029      1.053        773        800: 100% ━━━━━━━━━━━━ 1092/1092 1.8it/s 10:19
                 Class     Images  Instances   

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bc0666ed1c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0